# Conversational Threads 对话线程

许多 LLM 应用都采用类似聊天机器人的界面，用户与 LLM 应用进行多轮对话。要跟踪这些对话，可以使用 LangSmith 中的 `Threads` 功能。

这与 RAG 应用有关，该应用应该保留之前与用户对话的上下文。

### Setup

In [ ]:
# You can set them inline
# import os
# os.environ["OPENAI_API_KEY"] = "your openai api key"
# os.environ["LANGSMITH_API_KEY"] = "your langsmith api key"
# os.environ["LANGSMITH_TRACING"] = "true"
# os.environ["LANGSMITH_PROJECT"] = "langsmith-academy"  # If you don't set this, traces will go to the Default project

In [2]:
# Or you can use a .env file
from dotenv import load_dotenv
load_dotenv(dotenv_path="../../.env", override=True)

True

### Group traces into threads


线程是一系列轨迹，代表一次对话。每个回复都以单独的轨迹表示，但这些轨迹通过成为同一线程的一部分而相互关联。

要将跟踪关联起来，需要传入一个特殊的元数据键，其值是该线程的唯一标识符。

键值是该对话的唯一标识符。键名应为以下之一：

- session_id
- thread_id
- conversation_id.

The value should be a UUID.

In [3]:
import uuid
thread_id = uuid.uuid4()

In [4]:
from langsmith import traceable
from openai import OpenAI
from typing import List
import nest_asyncio
import os
from utils import get_vector_db_retriever

openai_client = OpenAI(
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
)
nest_asyncio.apply()
retriever = get_vector_db_retriever()

@traceable(run_type="chain")
def retrieve_documents(question: str):
    return retriever.invoke(question)

@traceable(run_type="chain")
def generate_response(question: str, documents):
    formatted_docs = "\n\n".join(doc.page_content for doc in documents)
    rag_system_prompt = """You are an assistant for question-answering tasks. 
    Use the following pieces of retrieved context to answer the latest question in the conversation. 
    If you don't know the answer, just say that you don't know. 
    Use three sentences maximum and keep the answer concise.
    """
    messages = [
        {
            "role": "system",
            "content": rag_system_prompt
        },
        {
            "role": "user",
            "content": f"Context: {formatted_docs} \n\n Question: {question}"
        }
    ]
    return call_openai(messages)

@traceable(run_type="llm")
def call_openai(
    messages: List[dict], model: str = "qwen3-max", temperature: float = 0.0
) -> str:
    return openai_client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature,
    )

@traceable(run_type="chain")
def langsmith_rag(question: str):
    documents = retrieve_documents(question)
    response = generate_response(question, documents)
    return response.choices[0].message.content


### 现在我们用这个 thread_id 运行两次应用程序。

In [5]:
question = "How do I add metadata to a Trace?"
ai_answer = langsmith_rag(question, langsmith_extra={"metadata": {"thread_id": thread_id}})
print(ai_answer)

To add metadata to a trace in the Python SDK, include a `metadata` dictionary as an argument to your `@traceable`-decorated function or pass it when calling `trace_with_attachments`. The metadata will be automatically associated with the trace. In TypeScript, you can include metadata in the options passed to the `traceable` wrapper alongside `extractAttachments`.


In [6]:
question = "How can I add tags to a Trace?"
ai_answer = langsmith_rag(question, langsmith_extra={"metadata": {"thread_id": thread_id}})
print(ai_answer)

You can add tags to a trace by including them in the `tags` parameter when creating a trace, either via the `@traceable` decorator or the `trace()` context manager. For example: `@traceable(tags=["tag1", "tag2"])` or `with trace(name="my_trace", tags=["tag1", "tag2"]) as run:`. Tags help categorize and filter traces in the LangSmith UI.


### Let's take a look in LangSmith!

<img src="../../images/threads.png">